# Pix2Struct widget-captioning-base — DIMER E2E UI widget captioning fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/pix2struct-ui-captioning-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/pix2struct-ui-captioning-pipeline/blob/main/tutorials/pix2struct_ui_captioning_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fpix2struct--widget--captioning--base-ffcc4d?style=flat)](https://huggingface.co/google/pix2struct-widget-captioning-base) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fpix2struct-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/pix2struct) [![arXiv](https://img.shields.io/badge/arXiv-2210.03347-b31b1b.svg)](https://arxiv.org/abs/2210.03347)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** UI widget captioning and bounded supervised fine-tuning of the caption decoder's last blocks on a screenshot/widget-box/reference-captions dataset, using the pinned `google/pix2struct-widget-captioning-base` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/pix2struct_ui_captioning_pipeline/`, at revision `e614ab6a241a`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `7e99642f87127dd3aee97ea82616bbfda5e610bb` (~1133 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `google/pix2struct-widget-captioning-base` snapshot (a 1.13 GB `model.safetensors`), downloads one digest-pinned parquet shard of Widget Captioning test widgets from the Hugging Face Hub (95 MB, no credential, refused on any size or SHA-256 mismatch), cuts a seeded subset of whole apps into training, validation and test widgets so no app or screen is shared, captions five boxed widgets on a drawn messaging-app screen through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test widgets with BLEU-4, ROUGE-L, CIDEr-D and unigram F1 beside the constant-caption and colour-nearest-neighbour baselines, runs a bounded fine-tuning of the caption decoder's last blocks with validation-CIDEr-D epoch selection, scores the held-out widgets again per category, re-captions the drawn screen with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify caption parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). A CUDA runtime is used automatically when present; the CPU path works but is slow (every widget draws its own box and is encoded at up to 2,048 patches), and the timings of the first clean run are recorded in `docs/release-verification.md`.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip holding a `records.jsonl` (or `records.json`) of `{{id, image, box, captions}}` objects — `image` a screenshot file name inside the zip, `box` the widget's `[x0, y0, x1, y1]` in pixels, `captions` one or more reference captions, optional `image_id`, `group` (the app, so its screens stay in one split) and `category` — beside the image files. They pass through the same validation, seeded app-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the Widget Captioning sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`google/pix2struct-widget-captioning-base` is the Pix2Struct model of Lee et al. (2023) — a ViT-style image encoder over variable-resolution 16×16 patches (up to 2,048 per image) and a 12-layer text decoder that cross-attends to them; 282,285,696 parameters, pretrained by parsing masked web screenshots into simplified HTML and fine-tuned on Widget Captioning, Android screenshots from the Rico corpus with human-written descriptions of individual UI elements — published under the **Apache-2.0** licence. It receives the screenshot with the **target widget outlined in blue** (the upstream preprocessing convention, reproduced by the carried module; no header text is rendered) and generates a short phrase for the widget's role (`search bar`, `go to profile`) with greedy decoding under a caller-owned `max_new_tokens` budget. **No score exists**: the caption is generated text with no probability and no correctness signal, and a fluent caption is **not evidence that it describes the boxed widget**.

What this notebook adds to inference is **adaptation with reference captions**. The dataset is real and from the checkpoint's own task: widgets from the Widget Captioning test split (Li et al., EMNLP 2020; **CC BY 4.0** as declared by the Hub mirror), whose apps the checkpoint was not trained on — so this is continued adaptation inside the task, and the honest question is a narrow one: does a bounded adaptation of the caption decoder's last blocks on a few hundred more widgets move the consensus metric on an app-disjoint test split at all, and on which kind of widget? Small widgets (mostly icons, under 1% of the screen) and larger ones (buttons, fields, rows) are read separately (`small-widget` / `large-widget`). The notebook downloads **one pinned parquet shard** (95 MB, SHA-256 pinned in the carried module) and draws a seeded subset of whole apps from it. Three captioning metrics are implemented in pure Python in the carried modules (**BLEU-4**, **ROUGE-L**, **CIDEr-D** — own implementations of the `coco-caption` definitions, with CIDEr-D's document frequencies taken from the evaluated set) beside the plumbing check `unigram_f1`, and two **non-neural baselines** — the corpus-medoid constant caption and a colour nearest neighbour over the widget's own crop — show where a system with no model sits. Nothing here is a quality claim about your screens: it is one seeded split of one shard.

**Weight-format note:** the pinned revision ships the model as SafeTensors (`model.safetensors`, digest-pinned in the manifest); the snapshot's processor declares the VQA variant, which the carried module switches off so that nothing but the box is added to the screenshot and no font is downloaded. Section 3 stages and digest-verifies the snapshot before the processor or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; download a digest-pinned shard of screenshots with widget boxes and reference captions, validate it and split it by app without leakage; caption through the public API over a drawn app screen and read `caption`, `box`, `new_tokens` and `truncated` correctly (generated text, no score); score the frozen model against several reference captions per widget with BLEU-4, ROUGE-L and CIDEr-D beside two non-neural baselines and read the `small-widget` / `large-widget` breakdown; run a bounded fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on an app-disjoint test split; re-caption a screen from a different image family with the adapted model; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** Widget detection (the caller supplies the box), screen summarisation or question answering (separate checkpoints), OCR of the screen, captions in languages other than English, batch throughput, sampling, beam search or repetition penalties (the notebook decodes greedily for reproducibility), SPICE (needs a scene-graph parser), evaluation on the Widget Captioning benchmark proper (only a seeded subset of one test shard is scored here), fine-tuning of the image encoder, the embeddings or the output projection, training on screens that are not the pinned sample or your own uploads, and any claim that a Rico split stands in for your app's screens. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available; a GPU runtime is recommended for Sections 6–8. Every widget draws its own box on its screenshot and is encoded at up to 2,048 patches, so captioning costs seconds per widget on CPU. The pinned `torch==2.14.0` install and the 1.13 GB checkpoint are the large downloads of the run; the widget shard adds 95 MB.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; what BLEU-4, ROUGE-L and CIDEr-D measure (n-gram precision with a brevity penalty, longest-common-subsequence F-measure, TF-IDF-weighted n-gram consensus) and why none is a human judgement; why a confident caption is not a correct one.
- **Data contract:** records are `{{id, image, box, captions}}` — a screenshot decodable by Pillow with sides between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px, a widget box `[x0, y0, x1, y1]` in pixels inside the image with sides of at least `MIN_BOX_SIDE` (4) px, and one or more non-empty reference captions of at most `MAX_CAPTION_CHARS` (200) characters (`MIN_CAPTIONS` = 1); optional `image_id` names the screen, optional `group` names the app (BYOD defaults it to the screen) and optional `category` labels the breakdown. Ids match `[A-Za-z0-9_.:-]{{1,64}}` and are unique; a dataset needs 8..5,000 records; every widget of the same app lands in the same split, so a test app's screens are never trained on; every (widget, reference caption) pair is one training target. BYOD accepts one zip of screenshots plus a `records.jsonl` / `records.json` in that shape.
- **Validation is structural, not semantic:** every screenshot is opened and decoded and every box and caption checked, but nothing checks that a reference caption describes the boxed widget — a mislabelled widget is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — app screenshots can show names, messages and account details. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path downloads one object from the Hub dataset repository `bevaya/RICO-WidgetCaptioning` at the immutable revision `6ec57b56…` (`data/test-00000-of-00002.parquet`, 95,313,640 bytes) and refuses it unless its size and SHA-256 match the pins carried in `samples.py`; only the screen id, caption, box, app-package and screenshot columns are read. The mirror declares CC BY 4.0 (Li et al., 2020, over Rico screens, Deka et al., 2017).
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/pix2struct-widget-captioning-base` snapshot (~1133 MB in total) at revision `7e99642f8712…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'pix2struct-ui-captioning-pipeline',
    'repository_revision': 'e614ab6a241a197e6709664a5bcb63321d6e968a',
    'embedded_module': 'src/pix2struct_ui_captioning_pipeline/pipeline.py',
    'embedded_modules': ['src/pix2struct_ui_captioning_pipeline/pipeline.py', 'src/pix2struct_ui_captioning_pipeline/metrics.py', 'src/pix2struct_ui_captioning_pipeline/samples.py'],
    'module_sha256': '93e726626fdf41ed41497e7326d7feaa2516d142ebb2aca8a0b4c7dba25b5c65',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/pix2struct_ui_captioning_pipeline/` @ `e614ab6a241a`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/pix2struct_ui_captioning_pipeline/pipeline.py`

In [ ]:
"""Widget captioning with the pinned ``google/pix2struct-widget-captioning-base`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The target widget is indicated
the way the upstream preprocessing did — a blue outline drawn on the screenshot, no header text — and
the model generates a short caption for it.

The adaptation contract (`evaluate`, `adapt`, `save_artifact`, `load_artifact`, `from_artifact`) fine-tunes
the caption decoder's last blocks on validated widget records with reference captions, selects the epoch on
validation CIDEr-D and exports the trained tensors as a safetensors adapter bound to the pinned base.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image, ImageDraw

MODEL_ID = "google/pix2struct-widget-captioning-base"
MODEL_REVISION = "7e99642f87127dd3aee97ea82616bbfda5e610bb"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "pix2struct-widget-captioning-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = "0b096c351692854237aa9fca73e973e3f764f376afc84999357614bac36e2b92"
PARAMETER_COUNT = 282_285_696  # 18,879,744 of them train by default (2 decoder blocks + final norm)
DECODER_LAYERS = 12
DEFAULT_TRAINABLE_DECODER_LAYERS = 2
# Evaluation bounds: a split larger than MAX_EVAL_RECORDS is refused (captioning is one encoder pass per
# widget); below MIN_SCORED_RECORDS the verdict says the sample is small.
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50
ARTIFACT_FORMAT = "org.valcorza.pix2struct-widget-captioning-base.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"

# Generation ceilings. Widget captions are a few words ("search bar", "go to profile"; the checkpoint's
# text_config max_length is 20); the default leaves room for a phrase, the ceiling bounds runaway
# generation.
MAX_NEW_TOKENS = 64
DEFAULT_MAX_NEW_TOKENS = 20
DECODING = "greedy"
# Widget box rendering: the upstream preprocessing (pix2struct/preprocessing/convert_widget_captioning.py)
# draws the target widget's bounds as a blue rectangle with a transparent fill and no header text.
BOX_COLOR = (0, 0, 255)
BOX_WIDTH = 3
MIN_BOX_SIDE = 4
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget (aspect ratio preserved), so pixel count only guards
# memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def normalize_caption(text: str) -> str:
    """COCO-caption-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def caption_tokens(text: str) -> list[str]:
    return normalize_caption(text).split()


def unigram_f1(prediction: str, references: Sequence[str]) -> float:
    """Bag-of-words F1 between the normalised prediction and the best-matching reference.

    A plumbing check, not a captioning metric: CIDEr, BLEU-4 and SPICE need several references per
    image and corpus-level statistics. Multiset overlap counts repeated words once per occurrence.
    """
    if not references:
        raise ValueError("references must contain at least one caption")
    pred = caption_tokens(prediction)
    best = 0.0
    for reference in references:
        ref = caption_tokens(reference)
        if not pred or not ref:
            continue
        ref_counts: dict[str, int] = {}
        for token in ref:
            ref_counts[token] = ref_counts.get(token, 0) + 1
        overlap = 0
        for token in pred:
            if ref_counts.get(token, 0) > 0:
                overlap += 1
                ref_counts[token] -= 1
        if overlap:
            precision, recall = overlap / len(pred), overlap / len(ref)
            best = max(best, 2 * precision * recall / (precision + recall))
    return best


def keyword_hits(caption: str, keywords: Sequence[str]) -> dict[str, bool]:
    """Which of the caller's keywords (normalised, whole-token match) appear in the caption."""
    tokens = set(caption_tokens(caption))
    return {keyword: all(part in tokens for part in caption_tokens(keyword)) for keyword in keywords}


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def validate_box(box: Any, image_size: tuple[int, int]) -> list[int]:
    """Check one widget box ``[x0, y0, x1, y1]`` in pixels: inside the image, at least MIN_BOX_SIDE a side."""
    if isinstance(box, (str, bytes)) or not isinstance(box, Sequence) or len(box) != 4:
        raise TypeError("box must be a sequence of four numbers [x0, y0, x1, y1]")
    values = []
    for value in box:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("box coordinates must be numbers")
        values.append(int(round(value)))
    x0, y0, x1, y1 = values
    width, height = image_size
    if x0 < 0 or y0 < 0 or x1 > width or y1 > height:
        raise ValueError(f"box {values} lies outside the {width}x{height} image")
    if x1 - x0 < MIN_BOX_SIDE or y1 - y0 < MIN_BOX_SIDE:
        raise ValueError(f"box {values} is smaller than MIN_BOX_SIDE {MIN_BOX_SIDE} px on a side")
    return values


def annotate_widget(image: Image.Image, box: Sequence[int]) -> Image.Image:
    """Return a copy of the RGB image with the widget outlined the way the upstream preprocessing did.

    Upstream (``pix2struct/preprocessing/convert_widget_captioning.py``) draws the target widget's
    bounds as a blue rectangle with a transparent fill and no header text; this package draws the same
    blue outline at BOX_WIDTH pixels.
    """
    rgb = validate_image(image)
    x0, y0, x1, y1 = validate_box(box, rgb.size)
    annotated = rgb.copy()
    ImageDraw.Draw(annotated).rectangle([x0, y0, x1, y1], outline=BOX_COLOR, width=BOX_WIDTH)
    return annotated


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one screenshot as PIL.Image.Image (any mode, converted to RGB) plus one widget box "
        "[x0, y0, x1, y1] in pixels"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "box": f"pixel coordinates inside the image, each side at least {MIN_BOX_SIDE} px",
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False, num_beams=1), deterministic on a fixed device and dtype",
    "preprocessing": (
        f"the widget box is drawn on a copy of the screenshot as a blue outline ({BOX_WIDTH} px, the "
        "upstream widget-captioning convention; no header text is rendered); the annotated screenshot is "
        "scaled to fill at most MAX_PATCHES 16x16 patches (aspect ratio preserved), normalised per image "
        "and flattened into patch tokens with row/column positions; the decoder generates the caption"
    ),
    "output": "one short caption string for the boxed widget (the model's decoded text), no score",
}


def _check_inputs(image: Any, box: Any, max_new_tokens: Any) -> tuple[Image.Image, list[int], int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``caption`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    checked_box = validate_box(box, rgb.size)
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_box, max_new_tokens


def validate_inputs(
    image: Image.Image,
    boxes: Sequence[Sequence[int]],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every box is checked exactly as ``caption`` would check it; rejection is reported by raising, and
    a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(boxes, (str, bytes)) or not isinstance(boxes, Sequence) or not boxes:
        raise TypeError("boxes must be a non-empty sequence of [x0, y0, x1, y1] boxes")
    if isinstance(boxes[0], (int, float)):
        raise TypeError("boxes must be a sequence of boxes, not a single box")
    checked = [_check_inputs(image, box, max_new_tokens)[1] for box in boxes]
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (caption takes one screenshot)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "boxes": checked,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    references: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``references`` (one sequence of reference captions per result, in order) the report carries
    the mean ``unigram_f1`` over the widgets plus one per-widget entry, verdict ``sample-sanity``;
    without references it is ``not-measurable`` and says what labelled data would make the task
    measurable. Neither is a captioning benchmark.
    """
    if not results:
        raise ValueError("results must contain at least one caption result")
    base = {
        "task": "screenshot + widget box -> short caption of the widget (widget captioning)",
        "score_semantics": (
            "the caption is generated text and carries no score, probability or correctness signal; a "
            "fluent caption is not evidence that it describes the boxed widget. Greedy decoding makes the "
            "output reproducible on a fixed device and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_widgets": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if references is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference captions were supplied for the captioned widgets",
            "needs": (
                "several human-written reference captions per widget from the deployment's own screens "
                "(Widget Captioning-style annotations) scored with CIDEr / BLEU-4 over a corpus; no such "
                "labelled set ships with this repository"
            ),
        }
    if len(references) != len(results):
        raise ValueError(f"references has {len(references)} entries for {len(results)} results")
    per_widget = []
    for result, refs in zip(results, references, strict=True):
        if isinstance(refs, str) or not refs:
            raise ValueError("each references entry must be a non-empty sequence of captions")
        prediction = str(result["caption"])
        per_widget.append(
            {
                "box": result.get("box"),
                "prediction": prediction,
                "references": list(refs),
                "unigram_f1": unigram_f1(prediction, refs),
            }
        )
    metrics = [
        {
            "id": "unigram_f1",
            "value": sum(entry["unigram_f1"] for entry in per_widget) / len(per_widget),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; best reference",
            "relation_to_benchmarks": (
                "bag-of-words overlap with the best reference; not CIDEr, BLEU-4 or SPICE, which need "
                "several references per widget and corpus-level statistics"
            ),
            "estimation": f"{len(per_widget)} widget(s) on one screenshot, no dispersion estimate",
        }
    ]
    return {
        **base,
        "metrics": metrics,
        "per_widget": per_widget,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_widget)} widget(s) with caller-written reference captions; plumbing evidence, not a "
            "captioning benchmark"
        ),
        "needs": (
            "several human-written reference captions per widget from the deployment's own screens scored "
            "with CIDEr / BLEU-4 over a corpus for any quality claim; the Widget Captioning dataset is not "
            "bundled"
        ),
    }


@dataclass
class Pix2StructWidgetCaptioningPipeline:
    """``_runner(annotated_image, max_new_tokens)`` returns ``{"caption": str, "new_tokens": int}``;
    injectable so the offline tests run without the model."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructWidgetCaptioningPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        return cls._from_model(model, processor, resolved_device, source)

    @classmethod
    def _from_model(
        cls, model: Any, processor: Any, device: str, source: str
    ) -> Pix2StructWidgetCaptioningPipeline:
        """Wrap a constructed model and processor (every parameter frozen, eval mode, header rendering off) in
        a pipeline; the offline tests use it with a small randomly initialised Pix2Struct model."""
        import torch

        # The snapshot's preprocessor_config declares is_vqa=True, which would render a text header
        # above the screenshot and fetch a font from the Hub. Upstream's widget-captioning preprocessing
        # renders no header - only the blue widget box - so the header path is disabled here; nothing
        # but the box is added to the image and no font is involved.
        processor.image_processor.is_vqa = False
        model = model.eval().to(device)
        for param in model.parameters():
            param.requires_grad_(False)

        def runner(annotated: Image.Image, max_new_tokens: int) -> dict[str, Any]:
            inputs = processor.image_processor(annotated, return_tensors="pt").to(device)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1
                )
            # Encoder-decoder: the output holds only decoder tokens (decoder_start + caption + eos).
            ids = generated[0]
            decoded = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
            return {"caption": decoded, "new_tokens": max(int(ids.shape[0]) - 1, 0)}

        return cls(runner, device, "float32", source, _model=model, _processor=processor)

    def caption(
        self,
        image: Image.Image,
        box: Sequence[int],
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Caption the widget inside ``box`` on one screenshot; ``caption`` is the decoded text, stripped."""
        rgb, checked_box, checked_tokens = _check_inputs(image, box, max_new_tokens)
        annotated = annotate_widget(rgb, checked_box)
        raw = self._runner(annotated, checked_tokens)
        if not isinstance(raw, dict) or "caption" not in raw:
            raise RuntimeError("runner must return a dict with 'caption'")
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "caption": str(raw["caption"]).strip(),
            "box": checked_box,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation contract ---------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._processor

    def predict(
        self, records: Sequence[Mapping[str, Any]], *, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS
    ) -> list[str]:
        """Caption every validated record's widget, in order."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        out = []
        for record in checked:
            with Image.open(record["image"]) as image:
                image.load()
                out.append(self.caption(image, record["box"], max_new_tokens=max_new_tokens)["caption"])
        return out

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS
    ) -> dict[str, Any]:
        """Caption every record's widget (greedy) and score the predictions against its reference captions
        (BLEU-4, ROUGE-L, CIDEr-D, unigram F1)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import caption_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = self.predict(checked, max_new_tokens=max_new_tokens)
        metrics = caption_metrics(predictions, [[str(c) for c in r["captions"]] for r in checked])
        metrics.update(
            {
                "max_new_tokens": max_new_tokens,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _decoder_layers(self) -> int:
        model, _ = self._require_model()
        return int(model.config.text_config.num_layers)

    def _trainable_names(self, trainable_decoder_layers: int) -> list[str]:
        """The last `trainable_decoder_layers` blocks of the caption decoder plus the decoder's final layer
        norm. The untied output projection (`decoder.lm_head`, vocabulary x hidden) and every embedding stay
        frozen, as does the whole image encoder."""
        if (
            isinstance(trainable_decoder_layers, bool)
            or not isinstance(trainable_decoder_layers, int)
            or not 1 <= trainable_decoder_layers <= DECODER_LAYERS
        ):
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        n_layers = self._decoder_layers()
        if trainable_decoder_layers > n_layers:
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{n_layers} for this model")
        first = n_layers - trainable_decoder_layers
        prefixes = tuple(f"decoder.layer.{k}." for k in range(first, n_layers))
        prefixes += ("decoder.final_layer_norm.",)
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def _encode_batch(self, records: Sequence[Mapping[str, Any]]) -> tuple[Any, Any]:
        """The frozen encoder's output and patch mask for a batch of boxed screenshots. The box is part of the
        image, so every widget is its own encoder input; it is recomputed per step under `no_grad` instead of
        cached (2,048 x 768 floats per widget)."""
        import torch

        model, processor = self._require_model()
        device = next(model.parameters()).device
        images = []
        for record in records:
            with Image.open(record["image"]) as image:
                images.append(annotate_widget(image.convert("RGB"), record["box"]))
        inputs = processor.image_processor(images, return_tensors="pt").to(device)
        with torch.no_grad():
            hidden = model.encoder(
                flattened_patches=inputs["flattened_patches"], attention_mask=inputs["attention_mask"]
            ).last_hidden_state
        return hidden, inputs["attention_mask"]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 3,
        lr: float = 1e-5,
        batch_size: int = 4,
        trainable_decoder_layers: int = DEFAULT_TRAINABLE_DECODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on validated widget records with reference captions.

        Only the last `trainable_decoder_layers` blocks of the caption decoder and the decoder's final layer
        norm train (2 blocks by default); the image encoder, every embedding and the untied output projection
        stay frozen. Each step takes `batch_size` widgets, draws their boxes on their screenshots exactly as
        `caption` does, runs the frozen encoder once per widget and trains on **every** (widget, reference
        caption) pair of the batch against that encoder output; the target is the tokenised caption with its
        end-of-sequence token, decoded with teacher forcing and scored with the model's own cross-entropy
        (padding ignored); AdamW at a fixed learning rate with gradient clipping at 1.0, no scheduler.
        Epoch 0 records the frozen model's validation metrics; the epoch with the highest validation CIDEr-D
        is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        names = self._trainable_names(trainable_decoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        import torch

        torch.manual_seed(seed)
        model, processor = self._require_model()
        tokenizer = processor.tokenizer
        pad_id = int(tokenizer.pad_token_id)
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = next(model.parameters()).device
        n_pairs = sum(len(r["captions"]) for r in train_checked)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked).items()
                if k in ("bleu4", "rouge_l", "cider_d", "unigram_f1", "mean_words", "n")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_score = entry["val"]["cider_d"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    chosen = [train_checked[j] for j in order[start : start + batch_size]]
                    hidden, mask = self._encode_batch(chosen)
                    owners = [i for i, r in enumerate(chosen) for _c in r["captions"]]
                    targets = tokenizer(
                        [str(c) for r in chosen for c in r["captions"]],
                        padding=True,
                        truncation=True,
                        max_length=MAX_NEW_TOKENS + 1,
                        return_tensors="pt",
                    ).to(device)
                    labels = targets["input_ids"].masked_fill(targets["input_ids"] == pad_id, -100)
                    index = torch.tensor(owners, device=hidden.device)
                    out = model(
                        encoder_outputs=(hidden.index_select(0, index),),
                        attention_mask=mask.index_select(0, index),
                        labels=labels,
                    )
                    optimiser.zero_grad(set_to_none=True)
                    out.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(out.loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["cider_d"] if entry["val"] else math.inf
                if current > best_score or not entry["val"]:
                    best_score = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_decoder_layers": trainable_decoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation CIDEr-D" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_pairs": n_pairs,
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted decoder-block and final-norm tensors as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported format
        and version, the pinned base (id, revision, weight file, digest), exactly one file entry named
        `adapter.safetensors` that resolves inside the artifact directory, and a recorded
        `trainable_decoder_layers` in range. Nothing is deserialised here. The digest check that follows
        detects corruption or drift of the weights relative to the adjacent manifest; it is not authenticity
        against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHT_FILE) != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_decoder_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int) or not 1 <= layers <= DECODER_LAYERS:
            raise ValueError("artifact manifest does not record an in-range integer trainable_decoder_layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_decoder_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("decoder."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable caption-decoder tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructWidgetCaptioningPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/pix2struct_ui_captioning_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level captioning metrics and two non-neural baselines, in pure Python.

`pipeline.py` keeps the per-caption plumbing check (`unigram_f1`); this module implements the three metrics a
captioning result is normally read by, over a set of records with several references each:

- **BLEU-4** (Papineni et al. 2002): corpus-level, clipped n-gram precision for n = 1..4 with geometric mean
  and brevity penalty against the closest reference length — the `coco-caption` convention.
- **ROUGE-L** (Lin 2004): per image the F-measure (beta = 1.2, as in `coco-caption`) of the longest common
  subsequence against the best-matching reference, averaged over images.
- **CIDEr-D** (Vedantam et al. 2015): TF-IDF-weighted n-gram cosine similarity for n = 1..4 with the
  Gaussian length penalty (sigma = 6) and clipping of candidate counts to the reference counts, averaged over
  references and n, scaled by 10. The document frequencies are computed over the references of the evaluated
  set, so a score is comparable only across systems evaluated on the same records (which is how it is used
  here: frozen vs. adapted vs. baselines on the same test split).

All three use COCO-style normalisation (`normalize_caption`: lower-case, punctuation removed). Two baselines a
fine-tuned model must beat: **constant caption** (the training caption that scores best against all other
training references — the corpus medoid, one string for every widget) and **colour nearest neighbour** (the
caption of the training widget whose 3x3 mean-colour grid over its box crop is closest — a lookup that knows
the widget only through 27 numbers).
"""

from __future__ import annotations

import math
from collections import Counter
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import caption_tokens, unigram_f1` removed — names are kernel globals defined by the carried modules

MAX_NGRAM = 4
ROUGE_BETA = 1.2
CIDER_SIGMA = 6.0
COLOUR_GRID = 3
METRIC_DEFINITIONS = {
    "bleu4": (
        "corpus-level BLEU-4: geometric mean of clipped 1..4-gram precisions against all references, times "
        "the brevity penalty against the closest reference length; in 0..1"
    ),
    "rouge_l": (
        "mean over images of the LCS-based F-measure (beta 1.2) against the best-matching reference; in 0..1"
    ),
    "cider_d": (
        "mean over images of the CIDEr-D consensus score: TF-IDF-weighted 1..4-gram cosine similarity to the "
        "references with a Gaussian length penalty (sigma 6), clipped counts, scaled by 10; document "
        "frequencies from the references of the evaluated set; typically 0..~1.5 on VizWiz"
    ),
    "unigram_f1": "mean over images of the bag-of-words F1 against the best-matching reference; in 0..1",
}


def _ngrams(tokens: Sequence[str], n: int) -> Counter[tuple[str, ...]]:
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def _check(predictions: Sequence[str], references: Sequence[Sequence[str]]) -> None:
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} predictions but {len(references)} reference lists")
    if not predictions:
        raise ValueError("no predictions to score")
    for index, refs in enumerate(references):
        if isinstance(refs, str) or not refs:
            raise ValueError(f"references[{index}] must be a non-empty list of captions")


def bleu4(predictions: Sequence[str], references: Sequence[Sequence[str]]) -> float:
    """Corpus-level BLEU-4 with multi-reference clipping and the closest-length brevity penalty."""
    _check(predictions, references)
    matched = [0] * MAX_NGRAM
    total = [0] * MAX_NGRAM
    hyp_len = ref_len = 0
    for prediction, refs in zip(predictions, references, strict=True):
        hyp = caption_tokens(prediction)
        ref_tokens = [caption_tokens(r) for r in refs]
        hyp_len += len(hyp)
        ref_len += min((abs(len(r) - len(hyp)), len(r)) for r in ref_tokens)[1]
        for n in range(1, MAX_NGRAM + 1):
            counts = _ngrams(hyp, n)
            max_ref: Counter[tuple[str, ...]] = Counter()
            for r in ref_tokens:
                for gram, count in _ngrams(r, n).items():
                    max_ref[gram] = max(max_ref[gram], count)
            matched[n - 1] += sum(min(count, max_ref[gram]) for gram, count in counts.items())
            total[n - 1] += max(len(hyp) - n + 1, 0)
    if any(m == 0 for m in matched) or hyp_len == 0:
        return 0.0
    log_precision = sum(math.log(m / t) for m, t in zip(matched, total, strict=True)) / MAX_NGRAM
    brevity = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / hyp_len)
    return brevity * math.exp(log_precision)


def _lcs(a: Sequence[str], b: Sequence[str]) -> int:
    if not a or not b:
        return 0
    previous = [0] * (len(b) + 1)
    for token in a:
        current = [0]
        for j, other in enumerate(b):
            current.append(previous[j] + 1 if token == other else max(previous[j + 1], current[j]))
        previous = current
    return previous[-1]


def rouge_l(prediction: str, references: Sequence[str]) -> float:
    """LCS F-measure (beta 1.2) against the best-matching reference."""
    hyp = caption_tokens(prediction)
    best = 0.0
    for reference in references:
        ref = caption_tokens(reference)
        lcs = _lcs(hyp, ref)
        if lcs == 0:
            continue
        precision, recall = lcs / len(hyp), lcs / len(ref)
        best = max(best, (1 + ROUGE_BETA**2) * precision * recall / (recall + ROUGE_BETA**2 * precision))
    return best


def _document_frequencies(references: Sequence[Sequence[str]]) -> list[Counter[tuple[str, ...]]]:
    """Per n, the number of images whose references contain each n-gram."""
    dfs: list[Counter[tuple[str, ...]]] = [Counter() for _ in range(MAX_NGRAM)]
    for refs in references:
        for n in range(1, MAX_NGRAM + 1):
            seen: set[tuple[str, ...]] = set()
            for r in refs:
                seen.update(_ngrams(caption_tokens(r), n))
            for gram in seen:
                dfs[n - 1][gram] += 1
    return dfs


def _tfidf(
    tokens: Sequence[str], n: int, df: Counter[tuple[str, ...]], log_images: float
) -> tuple[dict[tuple[str, ...], float], float]:
    counts = _ngrams(tokens, n)
    vector = {gram: count * (log_images - math.log(max(df[gram], 1.0))) for gram, count in counts.items()}
    return vector, math.sqrt(sum(v * v for v in vector.values()))


class _CiderIndex:
    """Document frequencies and reference TF-IDF vectors of one evaluated set, computed once."""

    def __init__(self, references: Sequence[Sequence[str]]) -> None:
        self.dfs = _document_frequencies(references)
        self.log_images = math.log(len(references))
        self.refs: list[list[tuple[list[str], list[tuple[dict[tuple[str, ...], float], float]]]]] = []
        for refs in references:
            entry = []
            for r in refs:
                tokens = caption_tokens(r)
                entry.append((tokens, [self._vector(tokens, n) for n in range(1, MAX_NGRAM + 1)]))
            self.refs.append(entry)

    def _vector(self, tokens: Sequence[str], n: int) -> tuple[dict[tuple[str, ...], float], float]:
        return _tfidf(tokens, n, self.dfs[n - 1], self.log_images)

    def score(self, prediction: str, image_index: int) -> float:
        hyp = caption_tokens(prediction)
        hyp_vectors = [self._vector(hyp, n) for n in range(1, MAX_NGRAM + 1)]
        refs = self.refs[image_index]
        total = 0.0
        for ref_tokens, ref_vectors in refs:
            penalty = math.exp(-((len(hyp) - len(ref_tokens)) ** 2) / (2 * CIDER_SIGMA**2))
            for (hyp_vec, hyp_norm), (ref_vec, ref_norm) in zip(hyp_vectors, ref_vectors, strict=True):
                if hyp_norm == 0 or ref_norm == 0:
                    continue
                dot = sum(min(hyp_vec[g], ref_vec[g]) * ref_vec[g] for g in hyp_vec if g in ref_vec)
                total += dot / (hyp_norm * ref_norm) * penalty
        return 10.0 * total / (MAX_NGRAM * len(refs))


def cider_d(predictions: Sequence[str], references: Sequence[Sequence[str]]) -> list[float]:
    """Per-image CIDEr-D scores with document frequencies from `references` (Vedantam et al. 2015,
    the `coco-caption` implementation)."""
    _check(predictions, references)
    index = _CiderIndex(references)
    return [index.score(p, i) for i, p in enumerate(predictions)]


def caption_metrics(predictions: Sequence[str], references: Sequence[Sequence[str]]) -> dict[str, Any]:
    """BLEU-4, ROUGE-L, CIDEr-D and unigram F1 over parallel predictions and reference lists."""
    _check(predictions, references)
    pairs = list(zip(predictions, references, strict=True))
    cider = cider_d(predictions, references)
    return {
        "n": len(predictions),
        "bleu4": bleu4(predictions, references),
        "rouge_l": sum(rouge_l(p, r) for p, r in pairs) / len(pairs),
        "cider_d": sum(cider) / len(cider),
        "unigram_f1": sum(unigram_f1(p, r) for p, r in pairs) / len(pairs),
        "empty_rate": sum(1 for p in predictions if not caption_tokens(p)) / len(predictions),
        "mean_words": sum(len(caption_tokens(p)) for p in predictions) / len(predictions),
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _references(records: Sequence[Mapping[str, Any]]) -> list[list[str]]:
    return [[str(c) for c in r["captions"]] for r in records]


def medoid_caption(train: Sequence[Mapping[str, Any]]) -> str:
    """The training caption with the highest mean CIDEr-D against every other training image's references."""
    if not train:
        raise ValueError("the constant-caption baseline needs training records")
    candidates = sorted({str(c) for r in train for c in r["captions"]})
    index = _CiderIndex(_references(train))
    best, best_score = candidates[0], -1.0
    for candidate in candidates:
        score = sum(index.score(candidate, i) for i in range(len(train))) / len(train)
        if score > best_score:
            best, best_score = candidate, score
    return best


def constant_caption_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """The same caption (the training medoid) for every image."""
    caption = medoid_caption(train)
    result = caption_metrics([caption] * len(records), _references(records))
    result["baseline"] = f"constant caption {caption!r}"
    return result


def colour_signature(image: str | Path | Image.Image, *, grid: int = COLOUR_GRID) -> list[float]:
    """Mean RGB of each cell of a `grid` x `grid` partition of the image, in 0..1 (27 numbers by default)."""
    handle = image if isinstance(image, Image.Image) else Image.open(image)
    with handle:
        small = handle.convert("RGB").resize((grid * 8, grid * 8), Image.BILINEAR)
        pixels = list(small.getdata())
    out: list[float] = []
    for row in range(grid):
        for col in range(grid):
            cell = [pixels[(row * 8 + y) * grid * 8 + col * 8 + x] for y in range(8) for x in range(8)]
            out.extend(sum(p[channel] for p in cell) / (64 * 255.0) for channel in range(3))
    return out


def _widget_crop(record: Mapping[str, Any]) -> Image.Image:
    """The record's widget: its box cropped from the screenshot (the whole image when no box is given)."""
    with Image.open(record["image"]) as handle:
        rgb = handle.convert("RGB")
    box = record.get("box")
    return rgb.crop(tuple(int(v) for v in box)) if box else rgb


def colour_neighbour_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """Caption every widget with the first reference of the training widget whose colour signature is
    closest (Euclidean distance over the 3x3 mean-colour grid of the box crop; no model)."""
    if not train:
        raise ValueError("the colour-neighbour baseline needs training records")
    signatures = [(colour_signature(_widget_crop(r)), str(r["captions"][0])) for r in train]
    predictions = []
    for record in records:
        query = colour_signature(_widget_crop(record))
        predictions.append(
            min(signatures, key=lambda s: sum((a - b) ** 2 for a, b in zip(s[0], query, strict=True)))[1]
        )
    result = caption_metrics(predictions, _references(records))
    result["baseline"] = f"colour nearest neighbour ({len(train)} training widgets)"
    return result

**Module 3/3:** `src/pix2struct_ui_captioning_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Widget-captioning dataset contract for fine-tuning: the pinned Widget Captioning sample, validation, seeded
app-disjoint splitting, BYOD loaders and JSONL export.

The default dataset is **real** and from the checkpoint's own task: widgets from the test split of Widget
Captioning (Li et al., EMNLP 2020; RICO screenshots, Deka et al. 2017), as mirrored on the Hugging Face Hub in
`bevaya/RICO-WidgetCaptioning` under **CC BY 4.0**. `google/pix2struct-widget-captioning-base` was fine-tuned
on Widget Captioning's *training* widgets, so this is continued adaptation inside the task on apps it was
not trained on, not a distribution shift. The sample is one pinned parquet shard
(`data/test-00000-of-00002.parquet`, 95,313,640 bytes) downloaded whole at the pinned dataset revision and
refused unless its SHA-256 matches the pin before `pyarrow` reads a byte of it; only the screen id, caption,
box, app-package and screenshot columns are read, and each screenshot is written to the cache once.

A record is ``{id, image_id, image, box, captions, group, category}`` — the path of the screenshot, the
widget box in pixels ``[x0, y0, x1, y1]`` (the corpus stores it relative to the image size), one or more
reference captions, the app package the screen belongs to, and a category (`small-widget` when the box covers
under `SMALL_WIDGET_AREA` of the screen — mostly icons — `large-widget` otherwise; BYOD records may carry any
label, `other` by default). Several widgets share a screen and several screens share an app; records of the
same app (`group`) are always kept in one split, as in the original benchmark split.

The shard's SHA-256 and the counts it yields are recorded by `tools/pin_corpus.py` (it needs Hub access);
until they are recorded, `fetch_corpus` refuses to read the shard rather than read an unpinned file.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import re
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MODEL_ID, validate_box, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Widget Captioning (RICO screens, test widgets)"
CORPUS_REPO = "bevaya/RICO-WidgetCaptioning"
CORPUS_REVISION = "6ec57b56bebd722b9c646c78d0f34e1199b6d7a9"
CORPUS_RELEASE = (
    "Widget Captioning test split (3,621 widgets) as mirrored on the Hugging Face Hub, "
    "dataset revision 6ec57b56"
)
CORPUS_LICENSE = "CC BY 4.0 (declared by the Hub mirror; Li et al. 2020 Widget Captioning over RICO screens)"
CORPUS_COLUMNS = ("screenId", "captions", "bbox", "app_package_name", "image")
# The shard's pins. `sha256`, `rows` and `screens` are written by tools/pin_corpus.py from a verified
# download; while `sha256` is None the reader refuses to run.
CORPUS_FILE: dict[str, Any] = {
    "path": "data/test-00000-of-00002.parquet",
    "bytes": 95_313_640,
    "sha256": "91d31536466cc5e6f1a15e284d766e80d1de0a94cd90bebe431135e1b51c9cb3",
    "rows": 1811,
    "screens": 646,
}
DEFAULT_CACHE_DIR = Path("weights") / "widget-captioning"
SAMPLE_SEED = 42
# Target widget counts per split; whole apps are allocated until each target is reached, so the realised
# counts can exceed a target by the widgets of the last app added.
SAMPLE_WIDGETS = {"train": 320, "validation": 64, "test": 160}
SMALL_WIDGET_AREA = 0.01
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MIN_CAPTIONS = 1
MAX_CAPTION_CHARS = 200
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def corpus_pinned() -> bool:
    """Whether the shard's SHA-256 has been recorded (see tools/pin_corpus.py)."""
    return isinstance(CORPUS_FILE.get("sha256"), str) and len(CORPUS_FILE["sha256"]) == 64


def _hub_download(cache: Path) -> Path:
    from huggingface_hub import hf_hub_download

    return Path(
        hf_hub_download(
            CORPUS_REPO,
            CORPUS_FILE["path"],
            repo_type="dataset",
            revision=CORPUS_REVISION,
            local_dir=str(cache),
        )
    )


def fetch_corpus(
    *, cache_dir: str | Path | None = None, downloader: Callable[[Path], Path] | None = None
) -> Path:
    """Return the path of the pinned shard, downloading it at the pinned revision when the cached copy is
    absent or drifted; refused on any size or SHA-256 mismatch, and outright while no pin is recorded."""
    if not corpus_pinned():
        raise RuntimeError(
            f"{CORPUS_REPO}@{CORPUS_REVISION[:8]} {CORPUS_FILE['path']}: no SHA-256 pin is recorded; run "
            "tools/pin_corpus.py with Hub access to record it before the sample can be read"
        )
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / CORPUS_FILE["path"]

    def ok(path: Path) -> bool:
        return (
            path.is_file()
            and path.stat().st_size == CORPUS_FILE["bytes"]
            and _sha256_file(path) == CORPUS_FILE["sha256"]
        )

    if ok(local):
        return local
    fetched = (downloader or _hub_download)(cache)
    if not ok(fetched):
        size = fetched.stat().st_size if fetched.is_file() else None
        raise ValueError(
            f"{CORPUS_FILE['path']}: fetched {size} bytes, pinned {CORPUS_FILE['bytes']} / "
            f"{CORPUS_FILE['sha256'][:16]}…; refusing to read it"
        )
    return fetched


def read_corpus(path: str | Path) -> list[dict[str, Any]]:
    """The shard's widgets as ``{screen_id, captions, bbox, app, image_bytes}`` (bbox relative, 0..1)."""
    import pyarrow.parquet as pq

    rows = pq.read_table(str(path), columns=list(CORPUS_COLUMNS)).to_pylist()
    out = []
    for index, row in enumerate(rows):
        image = row["image"]
        data = image.get("bytes") if isinstance(image, Mapping) else None
        if not data:
            raise ValueError(f"row {index}: no screenshot bytes")
        out.append(
            {
                "screen_id": str(row["screenId"]),
                "captions": [str(c) for c in row["captions"] or []],
                "bbox": [float(v) for v in row["bbox"]],
                "app": str(row["app_package_name"]),
                "image_bytes": bytes(data),
            }
        )
    if CORPUS_FILE.get("rows") is not None and len(out) != CORPUS_FILE["rows"]:
        raise ValueError(f"shard has {len(out)} rows, pinned {CORPUS_FILE['rows']}")
    return out


def widget_category(box: Sequence[int], image_size: Sequence[int]) -> str:
    """`small-widget` when the box covers under SMALL_WIDGET_AREA of the screen, else `large-widget`."""
    x0, y0, x1, y1 = box
    area = (x1 - x0) * (y1 - y0) / float(image_size[0] * image_size[1])
    return "small-widget" if area < SMALL_WIDGET_AREA else "large-widget"


def _pixel_box(bbox: Sequence[float], size: Sequence[int]) -> list[int]:
    width, height = size
    x0, y0, x1, y1 = bbox
    return [
        max(0, round(x0 * width)),
        max(0, round(y0 * height)),
        min(width, round(x1 * width)),
        min(height, round(y1 * height)),
    ]


def _image_suffix(fmt: str | None) -> str:
    return {"jpeg": ".jpg", "png": ".png", "webp": ".webp"}.get((fmt or "png").lower(), ".png")


def build_sample_dataset(
    rows: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
    image_dir: str | Path | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Group the widgets by app, shuffle the apps with `seed`, and allocate whole apps to test, validation
    and train until each split's widget target is reached. Widgets without a reference caption or whose
    box is out of range or under MIN_BOX_SIDE are skipped; the screenshots of the chosen widgets are written
    to `image_dir` under their screen id."""
    sizes = dict(sizes or SAMPLE_WIDGETS)
    out_dir = Path(image_dir) if image_dir is not None else DEFAULT_CACHE_DIR / "images"
    out_dir.mkdir(parents=True, exist_ok=True)
    if CORPUS_FILE.get("screens") is not None:
        screens = len({r["screen_id"] for r in rows})
        if screens != CORPUS_FILE["screens"]:
            raise ValueError(f"shard has {screens} distinct screens, pinned {CORPUS_FILE['screens']}")
    groups: dict[str, list[Mapping[str, Any]]] = {}
    for row in rows:
        groups.setdefault(str(row["app"]), []).append(row)
    order = sorted(groups)
    random.Random(seed).shuffle(order)
    out: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    written: dict[str, tuple[str, list[int]]] = {}
    for app in order:
        open_splits = [name for name in ("test", "validation", "train") if len(out[name]) < sizes[name]]
        if not open_splits:
            break
        target = open_splits[0]
        for row in groups[app]:
            captions = [" ".join(c.split()) for c in row["captions"] if c and c.strip()]
            if not captions:
                continue
            screen = str(row["screen_id"])
            if screen not in written:
                with Image.open(io.BytesIO(row["image_bytes"])) as image:
                    size, fmt = list(image.size), image.format
                path = out_dir / f"{screen}{_image_suffix(fmt)}"
                if not path.is_file() or _sha256_file(path) != _sha256_bytes(row["image_bytes"]):
                    path.write_bytes(row["image_bytes"])
                written[screen] = (str(path), size)
            path, size = written[screen]
            box = _pixel_box(row["bbox"], size)
            try:
                validate_box(box, (size[0], size[1]))
            except (TypeError, ValueError):
                continue
            out[target].append(
                {
                    "id": f"{target}-{len(out[target]):04d}",
                    "image_id": screen,
                    "image": path,
                    "box": box,
                    "captions": captions,
                    "group": str(app),
                    "category": widget_category(box, size),
                }
            )
    short = {name: (len(out[name]), sizes[name]) for name in out if len(out[name]) < sizes[name]}
    if short:
        raise ValueError(f"the shard's apps do not fill the split targets: {short}")
    return {"train": out["train"], "validation": out["validation"], "test": out["test"]}


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    downloader: Callable[[Path], Path] | None = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned shard."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    rows = read_corpus(fetch_corpus(cache_dir=cache, downloader=downloader))
    return build_sample_dataset(rows, seed=seed, sizes=sizes, image_dir=cache / "images")


def _check_record(record: Any, index: int, *, base_dir: Path | None) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/image/box/captions")
    for key in ("id", "image", "box", "captions"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    image_ref = record["image"]
    if not isinstance(image_ref, (str, Path)) or not str(image_ref).strip():
        raise ValueError(f"{label}: image must be a file path")
    path = Path(image_ref)
    if not path.is_absolute() and base_dir is not None:
        path = base_dir / path
    if not path.is_file():
        raise ValueError(f"{label}: image file not found: {path}")
    try:
        with Image.open(path) as handle:
            handle.load()
            validate_image(handle)
            width, height = handle.size
        box = validate_box(record["box"], (width, height))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: {exc}") from exc
    except OSError as exc:
        raise ValueError(f"{label}: image cannot be decoded: {exc}") from exc
    captions = record["captions"]
    if isinstance(captions, str) or not isinstance(captions, Sequence) or len(captions) < MIN_CAPTIONS:
        raise ValueError(f"{label}: captions must be a list of at least {MIN_CAPTIONS} reference caption(s)")
    checked_captions = [" ".join(str(c).split()) for c in captions]
    if not all(checked_captions):
        raise ValueError(f"{label}: every reference caption must be a non-empty string")
    if any(len(c) > MAX_CAPTION_CHARS for c in checked_captions):
        raise ValueError(f"{label}: a reference caption exceeds MAX_CAPTION_CHARS={MAX_CAPTION_CHARS}")
    image_id = str(record.get("image_id", path.name))
    return {
        "id": rid,
        "image_id": image_id,
        "image": str(path),
        "image_size": [width, height],
        "box": box,
        "captions": checked_captions,
        "group": str(record.get("group", image_id)),
        "category": str(record.get("category", "other")),
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    base_dir: str | Path | None = None,
) -> dict[str, Any]:
    """Structural validation of a widget-captioning dataset (every screenshot opened and decoded, every box
    held to the same checks `caption` applies); raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, image, box, captions} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    base = Path(base_dir) if base_dir is not None else None
    checked = []
    ids: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index, base_dir=base)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_images": len({r["image_id"] for r in checked}),
        "unique_groups": len({r["group"] for r in checked}),
        "categories": dict(Counter(r["category"] for r in checked)),
        "captions_per_widget": {
            "min": min(len(r["captions"]) for r in checked),
            "max": max(len(r["captions"]) for r in checked),
        },
        "caption_words": {
            "min": min(len(c.split()) for r in checked for c in r["captions"]),
            "max": max(len(c.split()) for r in checked for c in r["captions"]),
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [
        [
            r["id"],
            r["image_id"],
            list(r["box"]),
            list(r["captions"]),
            r.get("group", ""),
            r.get("category", ""),
        ]
        for r in records
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def reference_captions(record: Mapping[str, Any]) -> list[str]:
    """The reference captions of a record."""
    return [str(c) for c in record["captions"]]


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no app (`group`) and no screenshot appears in two splits (leakage check)."""
    seen: dict[tuple[str, str], str] = {}
    for name, records in splits.items():
        for record in records:
            image_id = str(record.get("image_id", record["id"]))
            for key in (("app", str(record.get("group", image_id))), ("screen", image_id)):
                if key in seen and seen[key] != name:
                    raise ValueError(f"{key[0]} {key[1]!r} appears in both {seen[key]} and {name}")
                seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
    base_dir: str | Path | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded split of a BYOD dataset into train/validation/test **by group** (app, defaulting to the
    screenshot): every widget of the same group lands in the same split."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, base_dir=base_dir)["records"]
    groups: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        groups.setdefault(record["group"], []).append(record)
    order = list(groups.values())
    random.Random(seed).shuffle(order)
    n_test = max(1, round(len(checked) * test_fraction))
    n_val = round(len(checked) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for group in order:
        if len(splits["test"]) < n_test:
            splits["test"].extend(group)
        elif len(splits["validation"]) < n_val:
            splits["validation"].extend(group)
        else:
            splits["train"].extend(group)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a JSON array or a JSONL file of ``{id, image, box, captions}`` objects; `image`
    paths are resolved relative to the file's directory by `validate_dataset(..., base_dir=...)`."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .json or .jsonl")


def write_dataset_jsonl(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One record per line in the shape `load_byod_dataset` reads back (image paths as given)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    keys = ("id", "image_id", "image", "box", "captions", "group", "category")
    with open(out, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps({k: record[k] for k in keys if k in record}, ensure_ascii=False) + "\n")
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `7e99642f8712…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Pix2StructWidgetCaptioningPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "pix2struct-widget-captioning-base",
  "modelId": "google/pix2struct-widget-captioning-base",
  "revision": "7e99642f87127dd3aee97ea82616bbfda5e610bb",
  "files": [
    {
      "path": "README.md",
      "bytes": 4499,
      "sha256": "82b161c0f6c74a1a520c6d3094927b5cb9f4163cd9a0bed390ee1974d91c3c4b"
    },
    {
      "path": "config.json",
      "bytes": 4904,
      "sha256": "301fe55f2135ea50cb3808d5112fa502d63e45a90b9ca0661e0d6e775f5a4b30"
    },
    {
      "path": "model.safetensors",
      "bytes": 1129177976,
      "sha256": "0b096c351692854237aa9fca73e973e3f764f376afc84999357614bac36e2b92"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 249,
      "sha256": "c84e4eebc84171d6069533d9f0147ec7b4afd02ab78697cb5c30f9419ef7dc45"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2583,
      "sha256": "5fdb6767a49aca48fdfa43d0279321918185fc4997bdb3ea72bf3a6301a1b43d"
    }
  ],
  "totalBytes": 1133308959
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Pix2StructWidgetCaptioningPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Widget Captioning screens, widgets and split

`fetch_corpus` returns the pinned shard from the cache under `weights/widget-captioning/` or downloads it at the pinned dataset revision, and refuses it unless its byte size and SHA-256 equal the pins in the carried module (it also refuses to run at all while no SHA-256 pin is recorded). `read_corpus` reads the screen id, captions, relative box, app package and screenshot columns with `pyarrow`. `build_sample_dataset` groups the widgets by app, shuffles the apps with `SPLIT_SEED` and allocates **whole apps** to the test, validation and training splits until each reaches its widget target (`SAMPLE_WIDGETS`), converting each box to pixels, skipping widgets with no caption or a box under `MIN_BOX_SIDE`, and labelling each `small-widget` (under 1% of the screen) or `large-widget`. `validate_dataset` then opens and decodes every screenshot and checks every record against the contract, `check_split_disjoint` asserts no app and no screen is shared, and the training split is written to `outputs/pix2struct_ui_captioning_train.jsonl` in the shape BYOD expects.

Look for: the shard's row and screen counts, the widget, screen and app counts per split, the category mix, captions per widget, three digests, and four refusal probes — a duplicate id, a missing image file, a box outside the screenshot and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import collections
import hashlib
import io
import json
import zipfile

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_dir = Path('work') / 'byod'
    byod_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            name = Path(member.filename).name
            if member.is_dir() or not name or name.startswith('.'):
                continue
            (byod_dir / name).write_bytes(archive.read(member))
    records_file = next(p for p in (byod_dir / 'records.jsonl', byod_dir / 'records.json') if p.is_file())
    records = load_byod_dataset(records_file)
    splits = split_dataset(records, seed=SPLIT_SEED, base_dir=byod_dir)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    shard_path = fetch_corpus(cache_dir='weights/widget-captioning')
    corpus_rows = read_corpus(shard_path)
    raw_rows = {'widgets': len(corpus_rows), 'screens': len({r['screen_id'] for r in corpus_rows}), 'apps': len({r['app'] for r in corpus_rows})}
    splits = build_sample_dataset(corpus_rows, seed=SPLIT_SEED, image_dir='weights/widget-captioning/images')
    data_source = f'{CORPUS_NAME} — {CORPUS_RELEASE} ({CORPUS_LICENSE})'
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
categories = {name: manifest['categories'] for name, manifest in dataset_manifests.items()}
write_dataset_jsonl(splits['train'], 'outputs/pix2struct_ui_captioning_train.jsonl')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'shard_sha256': str(CORPUS_FILE['sha256'])[:16] + '...'})
for name, manifest in dataset_manifests.items():
    print({name: {'widgets': manifest['n_records'], 'screens': manifest['unique_images'], 'apps': manifest['unique_groups'], 'categories': manifest['categories'], 'captions_per_widget': manifest['captions_per_widget'], 'caption_words': manifest['caption_words'], 'digest': manifest['digest'][:16] + '...'}})
example = splits['train'][0]
print({'example': {'id': example['id'], 'image': Path(example['image']).name, 'size': example['image_size'], 'box': example['box'], 'category': example['category'], 'captions': example['captions']}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in splits['train'][:8]],
    'missing image file': [{**splits['train'][0], 'image': 'work/does-not-exist.png'}, *splits['train'][1:8]],
    'box outside the screenshot': [{**splits['train'][0], 'box': [0, 0, 99999, 10]}, *splits['train'][1:8]],
    'too small': splits['train'][:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Caption through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: a flat mock of a messaging app drawn in code at 540×960 — a blue header reading `Messages` with a gear icon, a `Search conversations` field, three conversation rows with round avatars, a green `New message` button and a `Home / Chats / Calls / Profile` tab bar — with five widget boxes authored in pixel coordinates and a few **expected keywords** per widget. It is a different image family from the Rico screenshots, and the adapted model will be asked to caption the same boxes in Section 9. `validate_inputs` applies exactly the checks `caption` applies (image sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE`, each box inside the image with sides of at least `MIN_BOX_SIDE`, `max_new_tokens` in `[1, MAX_NEW_TOKENS]`) and returns an input manifest; a box that runs past the screenshot's edge is validated too and its rejection recorded as a finding. `caption` draws the blue outline on a copy and returns the decoded text, the checked box, `new_tokens`, a `truncated` flag and the model identity. **No score exists.** As recorded in the model card, the inference-only smoke captioned these five widgets `go to new message`, `search bar`, `go to next`, `select ana` and `profile` (the gear icon is the recorded miss). No reference captions are authored for the mock, so its `evaluation_report` is `not-measurable` by design and the keyword checks are observations; whether captions are *right* is what Section 6 measures on the test widgets with several human references each. The image digest depends on the Pillow build's bundled font rendering.

In [ ]:
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

CAPTION_MAX_TOKENS = 20  # @param {type:"integer"}


def synthetic_screen(width=540, height=960):
    """A flat messaging-app mock drawn with Pillow; returns image + [(widget name, box, expected keywords)]."""
    image = Image.new('RGB', (width, height), (245, 246, 250))
    d = ImageDraw.Draw(image)
    title, body = ImageFont.load_default(size=26), ImageFont.load_default(size=20)
    d.rectangle([0, 0, 540, 90], fill=(33, 90, 200))  # header bar
    d.text((30, 30), 'Messages', fill='white', font=title)
    d.rectangle([470, 25, 510, 65], outline='white', width=3)  # gear-like icon
    d.ellipse([482, 37, 498, 53], fill='white')
    d.rounded_rectangle([30, 120, 510, 170], radius=12, fill='white', outline=(200, 200, 200))  # search field
    d.text((50, 133), 'Search conversations', fill=(150, 150, 150), font=body)
    for index, (name, message) in enumerate([('Ana', 'See you at 6?'), ('Ben', 'Sent the files'), ('Cara', 'Happy birthday!')]):
        y = 200 + index * 90
        d.ellipse([30, y, 90, y + 60], fill=(120, 160, 220))  # avatar
        d.text((110, y + 5), name, fill='black', font=title)
        d.text((110, y + 38), message, fill=(110, 110, 110), font=body)
    d.rounded_rectangle([90, 820, 450, 880], radius=30, fill=(33, 150, 90))  # primary button
    d.text((270, 850), 'New message', fill='white', font=title, anchor='mm')
    d.rectangle([0, 900, 540, 960], fill='white')  # tab bar
    for index, label in enumerate(['Home', 'Chats', 'Calls', 'Profile']):
        d.text((67 + index * 135, 930), label, fill=(80, 80, 80), font=body, anchor='mm')
    widgets = [
        ('new-message button', [90, 820, 450, 880], ['message']),
        ('search field', [30, 120, 510, 170], ['search']),
        ('gear icon', [470, 25, 510, 65], ['settings']),  # smoke run: `go to next` (recorded miss)
        ('Ana avatar', [30, 200, 90, 260], ['ana']),
        ('Profile tab', [420, 905, 540, 955], ['profile']),
    ]
    return image, widgets


screen, widgets = synthetic_screen()
screen_name = 'synthetic_messaging_screen_540x960.png'
widget_names = [name for name, _, _ in widgets]
boxes = [box for _, box, _ in widgets]
expected_keywords = [keywords for _, _, keywords in widgets]
screen_sha256 = hashlib.sha256(np.asarray(screen.convert('RGB')).tobytes()).hexdigest()
ceilings = {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PATCHES': MAX_PATCHES, 'MIN_BOX_SIDE': MIN_BOX_SIDE, 'BOX_COLOR': BOX_COLOR, 'BOX_WIDTH': BOX_WIDTH, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'MIN_CAPTIONS': MIN_CAPTIONS, 'MAX_CAPTION_CHARS': MAX_CAPTION_CHARS}
print(ceilings)
input_manifest = validate_inputs(screen, boxes, max_new_tokens=CAPTION_MAX_TOKENS, names=[screen_name])
try:
    validate_inputs(screen, [[0, 0, screen.width + 20, 100]])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'box-outside-image-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/pix2struct_ui_captioning_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'screen': screen_name, 'rgb_sha256': screen_sha256[:16] + '...', 'boxes': len(boxes), 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})
results = []
for name, box, keywords in widgets:
    started = time.perf_counter()
    result = pipe.caption(screen, box, max_new_tokens=CAPTION_MAX_TOKENS)
    result['widget'] = name
    results.append({'seconds': round(time.perf_counter() - started, 3), **result})
    print(f"{name} {box}\n   caption: {result['caption']!r}  ({result['new_tokens']} tokens{', TRUNCATED' if result['truncated'] else ''})  keywords: {keyword_hits(result['caption'], keywords)}")
checks = {
    'one_result_per_widget': len(results) == len(widgets),
    'captions_are_text': all(isinstance(r['caption'], str) for r in results),
    'budget_respected': all(r['new_tokens'] <= CAPTION_MAX_TOKENS for r in results),
    'box_echoed': all(r['box'] == [int(v) for v in box] for r, box in zip(results, boxes, strict=True)),
}
if not all(checks.values()):
    raise RuntimeError(f'caption output failed a sanity check: {checks}')
frozen_screen = evaluation_report(results, None, sample_kind='synthetic')
frozen_keywords = [keyword_hits(r['caption'], k) for r, k in zip(results, expected_keywords, strict=True)]
print({'checks': checks, 'frozen_screen_verdict': frozen_screen['verdict'], 'keyword_hits': frozen_keywords})

## 6. Baselines and the frozen model's score on the test widgets

Three systems frame the adaptation, each read four ways. The **constant-caption baseline** answers every widget with the one training caption that scores highest against all other training references — the corpus medoid, a phrase that is safe everywhere and right nowhere. The **colour-nearest-neighbour baseline** answers with the first reference caption of the training widget whose 3×3 mean-colour grid over its box crop is closest — a lookup that knows the widget through 27 numbers. The **frozen model** captions every test widget with the budget from Section 5 and is scored with the same metrics: **BLEU-4** (corpus-level clipped n-gram precision with a brevity penalty), **ROUGE-L** (longest-common-subsequence F-measure against the best reference), **CIDEr-D** (TF-IDF-weighted n-gram consensus over all references, the metric Widget Captioning is ranked by) and the plumbing check **unigram F1**, all after lower-casing and punctuation removal. The checkpoint was fine-tuned on Widget Captioning's training apps, so expect it well above both baselines; the cell asserts only that it beats the constant caption. Read the per-category breakdown: icons (`small-widget`) carry no text for the model to read. The measured values of the first clean run are recorded in `docs/release-verification.md` and the model card.

In [ ]:
baseline_constant = constant_caption_baseline(train_records, test_records)
baseline_neighbour = colour_neighbour_baseline(train_records, test_records)
METRICS = ('bleu4', 'rouge_l', 'cider_d', 'unigram_f1')
print({'constant_caption_baseline': {k: round(baseline_constant[k], 3) for k in METRICS}, 'n': baseline_constant['n'], 'caption': baseline_constant['baseline']})
print({'colour_neighbour_baseline': {k: round(baseline_neighbour[k], 3) for k in METRICS}, 'note': baseline_neighbour['baseline']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, max_new_tokens=CAPTION_MAX_TOKENS)
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'mean_words': round(frozen_test['mean_words'], 1), 'n': frozen_test['n'], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})


def by_category(predictions, records):
    """CIDEr-D per category, with document frequencies from the whole evaluated set (as in `evaluate`)."""
    scores = cider_d(list(predictions), [reference_captions(r) for r in records])
    groups = collections.defaultdict(list)
    for record, score in zip(records, scores, strict=True):
        groups[record['category']].append(score)
    return {category: {'n': len(values), 'cider_d': round(sum(values) / len(values), 3)} for category, values in sorted(groups.items())}


medoid = medoid_caption(train_records)
constant_fields = by_category([medoid] * len(test_records), test_records)
frozen_predictions = pipe.predict(test_records, max_new_tokens=CAPTION_MAX_TOKENS)
frozen_fields = by_category(frozen_predictions, test_records)
print({'by_category': {'constant': constant_fields, 'frozen': frozen_fields}})
for record, prediction in list(zip(test_records, frozen_predictions, strict=True))[:3]:
    print({'category': record['category'], 'frozen': prediction, 'references': reference_captions(record)[:2]})
assert frozen_test['cider_d'] > baseline_constant['cider_d']

## 7. Bounded fine-tuning of the caption decoder's last blocks

`pipe.adapt` trains only the last `TRAINABLE_DECODER_LAYERS` blocks of the caption decoder plus the decoder's final layer norm — two blocks by default, 18,879,744 of 282,285,696 parameters; the image encoder, every embedding and the untied output projection (a 50,244 × 768 matrix) stay frozen. Each step takes `BATCH_SIZE` widgets, draws their boxes on their screenshots exactly as `caption` does, runs the frozen encoder once per widget (recomputed each step without gradients — the box makes every widget its own image) and trains on **every** (widget, reference caption) pair of the batch against that encoder output; the target is the tokenised caption with its end-of-sequence token, decoded with teacher forcing and scored with the model's own cross-entropy (padding ignored); AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Epoch 0 records the frozen model's validation metrics; every epoch is scored on the validation widgets, and the epoch with the highest validation CIDEr-D is kept — a validation split of about sixty widgets makes that selection noisy, which is why the held-out split in Section 8 is what the numbers are read from. If no epoch beats the frozen model on validation, the selector keeps epoch 0 and the adapter reproduces the frozen captions; that outcome is reported, not hidden.

In [ ]:
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE_DECODER_LAYERS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in METRICS})
        row['val_mean_words'] = round(entry['val']['mean_words'], 1)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_decoder_layers=TRAINABLE_DECODER_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'training_widgets': adapt_result['n_train'], 'training_pairs': adapt_result['n_pairs'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test widgets were never used for training or epoch selection, and no test app or screen appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, the four systems are put side by side on all four metrics, and the per-category CIDEr-D is repeated. Read it in this order: **CIDEr-D** first (the consensus metric the epoch was selected on), then BLEU-4 and ROUGE-L, which can move the other way when the adapted captions change length, then the `small-widget` / `large-widget` split. `adapted_beats_frozen` records whether the held-out CIDEr-D rose. About 160 widgets from one seeded draw of one shard give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on Rico apps says nothing about your app until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records, max_new_tokens=CAPTION_MAX_TOKENS)
adapted_val = pipe.evaluate(val_records, max_new_tokens=CAPTION_MAX_TOKENS)
adapted_predictions = pipe.predict(test_records, max_new_tokens=CAPTION_MAX_TOKENS)
adapted_fields = by_category(adapted_predictions, test_records)
comparison = {metric: {'constant': round(baseline_constant[metric], 3), 'neighbour': round(baseline_neighbour[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['mean_words'] = {'constant': round(baseline_constant['mean_words'], 1), 'neighbour': round(baseline_neighbour['mean_words'], 1), 'frozen': round(frozen_test['mean_words'], 1), 'adapted': round(adapted_test['mean_words'], 1)}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
comparison['by_category'] = {category: {'n': frozen_fields[category]['n'], 'constant': constant_fields[category]['cider_d'], 'frozen': frozen_fields[category]['cider_d'], 'adapted': adapted_fields[category]['cider_d']} for category in frozen_fields}
for key, row in comparison.items():
    print({key: row})
for record, before, after in list(zip(test_records, frozen_predictions, adapted_predictions, strict=True))[:3]:
    print({'category': record['category'], 'frozen': before, 'adapted': after, 'references': reference_captions(record)[:2]})
adapted_beats_frozen = adapted_test['cider_d'] > frozen_test['cider_d']
print({'adapted_beats_frozen': adapted_beats_frozen})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'categories': categories,
    'max_new_tokens': CAPTION_MAX_TOKENS,
    'baselines': {'constant_caption': baseline_constant, 'colour_neighbour': baseline_neighbour},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
    'adapted_beats_frozen': adapted_beats_frozen,
}
with open('outputs/pix2struct_ui_captioning_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
print({'report': 'outputs/pix2struct_ui_captioning_evaluation_report.json'})

## 9. Re-caption the drawn screen, export the adapter and reload it

The five widgets on the drawn screen from Section 5 are captioned again by the adapted model and their keyword observations repeated — a mock from a different image family than the Rico screenshots it was tuned on, so this is a small look at whether the adaptation changed the model's behaviour *outside* its sample (five widgets of evidence, not a measurement; a different caption here is a finding to record, not a failure). Both caption sets are written as CSV.

`pipe.save_artifact` writes the trained tensors — the caption decoder's last two blocks and its final layer norm, about 76 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `Pix2StructWidgetCaptioningPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor that is not a caption-decoder tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical captions on eight test widgets (VER4).

In [ ]:
import csv
import shutil

adapted_results = []
for name, box, keywords in widgets:
    result = pipe.caption(screen, box, max_new_tokens=CAPTION_MAX_TOKENS)
    result['widget'] = name
    adapted_results.append(result)
adapted_screen = evaluation_report(adapted_results, None, sample_kind='synthetic')
adapted_keywords = [keyword_hits(r['caption'], k) for r, k in zip(adapted_results, expected_keywords, strict=True)]
for before, after, hits in zip(results, adapted_results, adapted_keywords, strict=True):
    print({'widget': before['widget'], 'frozen': before['caption'], 'adapted': after['caption'], 'adapted_keywords': hits})
with open('outputs/pix2struct_ui_captioning_captions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'widget', 'box', 'frozen_caption', 'adapted_caption', 'adapted_new_tokens', 'expected_keywords'])
    for before, after, keywords in zip(results, adapted_results, expected_keywords, strict=True):
        writer.writerow([screen_name, before['widget'], ' '.join(str(v) for v in before['box']), before['caption'], after['caption'], after['new_tokens'], ' | '.join(keywords)])

artifact_dir = Path('outputs/pix2struct_ui_captioning_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'pix2struct_ui_captioning', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = Pix2StructWidgetCaptioningPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = pipe.predict(test_records[:8], max_new_tokens=CAPTION_MAX_TOKENS)
after = reloaded.predict(test_records[:8], max_new_tokens=CAPTION_MAX_TOKENS)
parity = {'identical_captions': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_captions'] == parity['of']

weight_entry = next(entry for entry in MANIFEST['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'SafeTensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'file': CORPUS_FILE, 'sample_widgets': SAMPLE_WIDGETS},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'screen': {'name': screen_name, 'size': list(screen.size), 'rgb_sha256': screen_sha256, 'widgets': widget_names, 'boxes': boxes, 'expected_keywords': expected_keywords}, 'items': [{k: r[k] for k in ('widget', 'caption', 'new_tokens', 'truncated', 'seconds')} for r in results], 'frozen_report': frozen_screen, 'frozen_keywords': frozen_keywords, 'adapted_items': [{k: r[k] for k in ('widget', 'caption', 'new_tokens', 'truncated')} for r in adapted_results], 'adapted_report': adapted_screen, 'adapted_keywords': adapted_keywords},
    'comparison': comparison,
    'adapted_beats_frozen': adapted_beats_frozen,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/pix2struct_ui_captioning_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen model is a Widget Captioning checkpoint scored on apps it was not trained on, beside two non-neural baselines, and a bounded fine-tuning of the caption decoder's last two blocks on a few hundred more widgets is then scored on an app-disjoint test split — overall, and separately on small and larger widgets — with an adapter that reloads to identical captions. That is the claim: the adaptation contract works end to end on a real widget-captioning corpus, and the numbers it produces are read on four metrics and per category against two non-neural baselines and the frozen model rather than in isolation. Whether the held-out CIDEr-D rose is recorded as `adapted_beats_frozen`, not assumed.

The test split is about 160 widgets on whole apps from one seeded draw of one shard, the validation split that picks the epoch is about 60, the metrics are four reference-based scores (own pure-Python implementations of the `coco-caption` definitions, with CIDEr-D's document frequencies from the evaluated set — so its absolute value is not comparable to the benchmark's published numbers — and none a human judgement). Because the checkpoint already saw Widget Captioning's training apps, a small or zero gain is the expected outcome and not a failure of the contract; a learning rate that is too high overfits this little data within an epoch, which the validation-based selector reports by keeping epoch 0. So a gain here says the contract works, not that the adapted model describes your app's widgets better; it still captions every box — including one around nothing — with a fluent phrase. Fine-tuning on a narrow sample can also erode the model elsewhere; the drawn screen re-captioned in Section 9 is five widgets of evidence about that, not a measurement.

Three things to carry to real data. **Baselines first:** the constant-caption and colour-neighbour baselines and the frozen model's score on *your* references are the numbers to read before any adapted one, per category and on CIDEr-D. **Leakage:** keep every widget of an app in one split (the contract does this), never split at random over screens of the same app. **The box is part of the input:** a wrong or loose box is a wrong request, and the model will caption it anyway.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real widget-captioning corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against two trivial baselines and the frozen model on an app-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, caption quality on any other app population or platform, or production fitness.

**Optional experiments (they do not affect the default path):** raise `LEARNING_RATE` and watch the training loss fall while the validation CIDEr-D drops and the selector keeps an early epoch; set `TRAINABLE_DECODER_LAYERS = 1` and compare the artifact size and the held-out score; loosen a box on the drawn screen and watch the caption follow it; or bring your own screenshots through BYOD and read the two baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/pix2struct-ui-captioning-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/pix2struct-ui-captioning-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/pix2struct-ui-captioning-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model (Google, Apache-2.0): https://huggingface.co/google/pix2struct-widget-captioning-base
- Upstream code: https://github.com/google-research/pix2struct
- Pix2Struct: Screenshot Parsing as Pretraining for Visual Language Understanding (Lee et al., ICML 2023): https://arxiv.org/abs/2210.03347
- Widget Captioning: Generating Natural Language Description for Mobile User Interface Elements (Li et al., EMNLP 2020): https://arxiv.org/abs/2010.04295 — data: https://github.com/google-research-datasets/widget-caption
- Rico: A Mobile App Dataset for Building Data-Driven Design Applications (Deka et al., UIST 2017): https://dl.acm.org/doi/10.1145/3126594.3126651
- Widget Captioning as mirrored on the Hugging Face Hub (CC BY 4.0): https://huggingface.co/datasets/bevaya/RICO-WidgetCaptioning
- BLEU: a Method for Automatic Evaluation of Machine Translation (Papineni et al., 2002): https://aclanthology.org/P02-1040/
- ROUGE: A Package for Automatic Evaluation of Summaries (Lin, 2004): https://aclanthology.org/W04-1013/
- CIDEr: Consensus-based Image Description Evaluation (Vedantam et al., 2015): https://arxiv.org/abs/1411.5726
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)